# 09 - Decision Engine + Pipeline Orchestrator

Stage 9: Decision Engine + Pipeline Orchestrator.

This is the piece that actually connects your crop identifier to the
right disease classifier, using the shared context-object pattern from
the architecture plan (each stage reads/writes the same dict, with a
status field tracking progress).

Gracefully handles crops that don't have a trained disease model yet
(Groundnut, Pepper Bell, Potato as of now) -- routes correctly, reports
"not yet trained" instead of crashing, so you can run this end-to-end
right now with just Cotton + Tomato and it'll keep working as you train
the rest.

Install deps: same as 05_test_ui.py
    pip install torch timm opencv-python albumentations --break-system-packages

## Imports & Configuration

In [1]:
import json
from pathlib import Path

import cv2
import numpy as np
import torch
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

MODELS_DIR = Path(r"Z:\Projects\Smart-Farming\models")
IMG_SIZE = 224
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

eval_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# Confidence floors -- below these, the pipeline stops and flags the
# result as unreliable rather than pushing a low-confidence guess
# through to the next stage.
CROP_CONFIDENCE_THRESHOLD = 0.75
DISEASE_CONFIDENCE_THRESHOLD = 0.60

_model_cache = {}  # avoids reloading a model on every single prediction

c:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## `compute_sharpness_map`

In [2]:
def compute_sharpness_map(gray, ksize=25):
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return cv2.blur(lap ** 2, (ksize, ksize))

## `green_mask`

In [3]:
def green_mask(hsv):
    mask = cv2.inRange(hsv, np.array([25, 30, 30]), np.array([95, 255, 255]))
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    return mask

## `score_contour`

In [4]:
def score_contour(contour, image_shape, sharpness_map):
    h, w = image_shape[:2]
    area = cv2.contourArea(contour)
    if area < 0.005 * h * w:
        return -1, None
    x, y, bw, bh = cv2.boundingRect(contour)
    cx, cy = x + bw / 2, y + bh / 2
    dist = np.hypot(cx - w / 2, cy - h / 2)
    centrality = 1 - (dist / np.hypot(w / 2, h / 2))
    region_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.drawContours(region_mask, [contour], -1, 255, thickness=cv2.FILLED)
    sharpness = cv2.mean(sharpness_map, mask=region_mask)[0]
    score = (0.45 * area / (h * w)) + (0.30 * centrality) + (0.25 * min(sharpness / 500, 1.0))
    return score, (x, y, bw, bh)

## `isolate_subject_leaf`

In [5]:
def isolate_subject_leaf(image_rgb, padding_ratio=0.08):
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2HSV)
    sharpness_map = compute_sharpness_map(gray)
    contours, _ = cv2.findContours(green_mask(hsv), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    best_score, best_box = -1, None
    for c in contours:
        s, box = score_contour(c, image_rgb.shape, sharpness_map)
        if s > best_score:
            best_score, best_box = s, box
    if best_box is None or best_score < 0.15:
        return None
    x, y, bw, bh = best_box
    pad_x, pad_y = int(bw * padding_ratio), int(bh * padding_ratio)
    h, w = image_rgb.shape[:2]
    x0, y0 = max(0, x - pad_x), max(0, y - pad_y)
    x1, y1 = min(w, x + bw + pad_x), min(h, y + bh + pad_y)
    return image_rgb[y0:y1, x0:x1]

## `_load_model`

In [6]:
def _load_model(model_path, labels_path, arch):
    key = str(model_path)
    if key in _model_cache:
        return _model_cache[key]
    if not model_path.exists() or not labels_path.exists():
        return None
    with open(labels_path) as f:
        classes = json.load(f)
    model = timm.create_model(arch, pretrained=False, num_classes=len(classes))
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.to(DEVICE)
    model.eval()
    _model_cache[key] = (model, classes)
    return _model_cache[key]

## `get_crop_model`

In [7]:
def get_crop_model():
    return _load_model(
        MODELS_DIR / "crop_identifier_v1.pth",
        MODELS_DIR / "crop_identifier_labels.json",
        "efficientnet_b0",
    )

## `get_disease_model`

The Decision Engine's core job: crop label -> the right disease model.

In [8]:
def get_disease_model(crop_label):
    """The Decision Engine's core job: crop label -> the right disease model."""
    crop_slug = crop_label.replace(" ", "_")
    return _load_model(
        MODELS_DIR / f"disease_{crop_slug}.pth",
        MODELS_DIR / f"disease_{crop_slug}_labels.json",
        "efficientnet_b2",
    )

## `predict_with_model`

In [9]:
def predict_with_model(model, classes, image_rgb):
    tensor = eval_tf(image=image_rgb)["image"].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(tensor), dim=1)[0].cpu().numpy()
    top_idx = int(np.argmax(probs))
    return classes[top_idx], float(probs[top_idx]), {classes[i]: float(probs[i]) for i in range(len(classes))}

## `new_context`

In [10]:
def new_context(image_rgb):
    return {
        "image": {"raw": image_rgb, "leaf_crop": None},
        "crop": {"label": None, "confidence": None},
        "disease": {"label": None, "confidence": None, "model_used": None},
        "status": {
            "preprocessing": "pending",
            "crop_identification": "pending",
            "decision_routing": "pending",
            "disease_classification": "pending",
        },
        "notes": [],
    }

## `preprocess`

In [11]:
def preprocess(context):
    leaf_crop = isolate_subject_leaf(context["image"]["raw"])
    if leaf_crop is None:
        context["notes"].append("No confident leaf region found -- used full image.")
        leaf_crop = context["image"]["raw"]
    context["image"]["leaf_crop"] = leaf_crop
    context["status"]["preprocessing"] = "completed"
    return context

## `identify_crop`

In [12]:
def identify_crop(context):
    result = get_crop_model()
    if result is None:
        context["notes"].append("Crop identifier model not found at models/crop_identifier_v1.pth.")
        context["status"]["crop_identification"] = "failed"
        return context
    model, classes = result
    label, confidence, _ = predict_with_model(model, classes, context["image"]["leaf_crop"])
    context["crop"]["label"] = label
    context["crop"]["confidence"] = confidence
    context["status"]["crop_identification"] = "completed"
    if confidence < CROP_CONFIDENCE_THRESHOLD:
        context["notes"].append(
            f"Crop confidence ({confidence:.2f}) below threshold ({CROP_CONFIDENCE_THRESHOLD}) -- "
            f"stopping before disease classification. Consider a clearer photo."
        )
        context["status"]["decision_routing"] = "skipped_low_confidence"
    return context

## `route_to_disease_model`

In [13]:
def route_to_disease_model(context):
    if context["status"]["crop_identification"] != "completed":
        context["status"]["decision_routing"] = "skipped"
        return context
    if context["crop"]["confidence"] < CROP_CONFIDENCE_THRESHOLD:
        return context  # already flagged in identify_crop

    result = get_disease_model(context["crop"]["label"])
    if result is None:
        context["notes"].append(
            f"No disease model trained yet for '{context['crop']['label']}' -- "
            f"skipping disease classification for this crop."
        )
        context["status"]["decision_routing"] = "no_model_available"
        return context

    context["status"]["decision_routing"] = "completed"
    context["_disease_model_bundle"] = result  # internal, not part of the public result
    return context

## `classify_disease`

In [14]:
def classify_disease(context):
    if context["status"]["decision_routing"] != "completed":
        context["status"]["disease_classification"] = "skipped"
        return context
    model, classes = context.pop("_disease_model_bundle")
    label, confidence, all_probs = predict_with_model(model, classes, context["image"]["leaf_crop"])
    context["disease"]["label"] = label
    context["disease"]["confidence"] = confidence
    context["disease"]["model_used"] = f"disease_{context['crop']['label'].replace(' ', '_')}"
    context["status"]["disease_classification"] = "completed"
    if confidence < DISEASE_CONFIDENCE_THRESHOLD:
        context["notes"].append(
            f"Disease confidence ({confidence:.2f}) is low -- treat this prediction as tentative."
        )
    return context

## `run_pipeline`

In [15]:
def run_pipeline(image_rgb):
    context = new_context(image_rgb)
    context = preprocess(context)
    context = identify_crop(context)
    context = route_to_disease_model(context)
    context = classify_disease(context)
    return context

## Run

In [ ]:
# Quick manual test -- point this at any leaf photo you have on disk.
test_image_path = r"Z:\Projects\Smart-Farming\Datasets\generated_crops\groundnut\leaf_crops\GLUE_ELS_001.jpg"
image_bgr = cv2.imread(test_image_path)
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

result = run_pipeline(image_rgb)
print(f"Crop: {result['crop']['label']} (confidence {result['crop']['confidence']})")
print(f"Disease: {result['disease']['label']} (confidence {result['disease']['confidence']})")
print(f"Status: {result['status']}")
if result["notes"]:
    print("Notes:", result["notes"])

Crop: Groundnut (confidence 0.9999983310699463)
Disease: Leaf Spot (confidence 0.9999765157699585)
Status: {'preprocessing': 'completed', 'crop_identification': 'completed', 'decision_routing': 'completed', 'disease_classification': 'completed'}


: 